# 🤖📰 Treinar a IA de Notícias — SmartTrader

Treina/usa a **IA que lê notícias e decide direção** (comprar/vender). Rode no **Colab** com **GPU**.

### ⚠️ Honestidade primeiro
- **FinBERT** mede **sentimento** (já vem treinado). 'Treinar' = especializar na GPU.
- Sentimento **não é direção** — quem vira 'petróleo↑' em 'USDCAD↓' é a camada macro (Passo 4).
- Não garante lucro. Valide em demo antes de real.

### 🛠️ Erros comuns
- **Nunca reinstale `torch`** (o Colab já tem um compatível).
- Usamos um dataset em **parquet** (sem 'script'), então não precisa fixar versão nem reiniciar.


## Passo 1 — Instalar dependências (sem torch)


In [ ]:
!pip install -q -U transformers datasets
import torch
print('Torch:', torch.__version__, '| GPU:', torch.cuda.is_available())


## Passo 2 — Usar o FinBERT pronto (já funciona)
Carrega o modelo direto (sem `pipeline`, que puxa o torchvision e dá conflito).


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL = 'ProsusAI/finbert'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL); model.eval()

def sentimento(texto):
    inp = tok(texto, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad(): logits = model(**inp).logits
    probs = torch.softmax(logits, dim=1)[0]; i = int(probs.argmax())
    return model.config.id2label[i], float(probs[i])

for m in ['Oil prices spike after OPEC supply cut amid Middle East war',
          'Tech stocks rally as inflation cools and growth beats forecasts',
          'Markets plunge as recession fears and conflict escalate']:
    lab, sc = sentimento(m)
    print(f'{lab:9s} ({sc:.2f})  <-  {m}')


## Passo 3 — (GPU) Fine-tunar num dataset financeiro REAL (parquet)
Usa `zeroshot/twitter-financial-news-sentiment` (notícias financeiras rotuladas,
formato parquet — sem script, funciona direto). 3 classes. Opcional.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np

# Dataset financeiro REAL em parquet (sem 'script' -> funciona em qualquer versao do datasets)
d = load_dataset('zeroshot/twitter-financial-news-sentiment')
train_ds, eval_ds = d['train'], d['validation']
print('Colunas:', train_ds.column_names)
text_col = 'text' if 'text' in train_ds.column_names else next(c for c in train_ds.column_names if c != 'label')

MODEL = 'ProsusAI/finbert'
tok = AutoTokenizer.from_pretrained(MODEL)
def prep(b): return tok(b[text_col], truncation=True, padding='max_length', max_length=128)
train_ds = train_ds.map(prep, batched=True).rename_column('label', 'labels')
eval_ds  = eval_ds.map(prep, batched=True).rename_column('label', 'labels')
cols = ['input_ids', 'attention_mask', 'labels']
train_ds.set_format('torch', columns=cols); eval_ds.set_format('torch', columns=cols)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3, ignore_mismatched_sizes=True)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': float((preds == p.label_ids).mean())}

args = TrainingArguments(output_dir='/content/finbert_ft', num_train_epochs=1,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy='epoch', logging_steps=100, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
    eval_dataset=eval_ds, compute_metrics=metrics)
trainer.train(); print('Avaliação:', trainer.evaluate())
trainer.save_model('/content/finbert_ft'); tok.save_pretrained('/content/finbert_ft')
print('Modelo salvo em /content/finbert_ft')


## Passo 4 — De SENTIMENTO para DIREÇÃO (a sua ideia)
A notícia vira **comprar/vender** por ativo, tratando **forças conflitantes**.
Versão resumida do `news_mapper.py` do projeto.


In [ ]:
THEME_KW = {'oil':['oil','crude','opec','brent','wti','petroleum','petroleo'],
  'risk_off':['war','crisis','conflict','recession','crash','plunge'],
  'risk_on':['rally','optimism','growth beats','soar']}
EXPO = {'USDCAD':{'oil':-0.7,'risk_off':+0.6},'XAUUSD':{'risk_off':+0.8,'oil':+0.2},
  'USOIL':{'oil':+1.0},'SPX':{'risk_off':-0.8,'risk_on':+0.7}}

def detectar_temas(textos):
    t=' '.join(textos).lower(); temas={}
    for tema,kws in THEME_KW.items():
        hits=sum(t.count(k) for k in kws)
        if hits: temas[tema]=min(1.0, hits/(hits+1))
    return temas

def direcao(simbolo, temas):
    net=sum(EXPO.get(simbolo,{}).get(tm,0.0)*f for tm,f in temas.items())
    return (1 if net>0.15 else (-1 if net<-0.15 else 0)), min(1.0,abs(net)), net

noticia=['Oil prices spike after OPEC supply cut amid Middle East war']
lab, sc = sentimento(noticia[0])
print('Sentimento FinBERT:', lab, f'({sc:.2f})')
temas=detectar_temas(noticia); print('Temas:', temas)
for s in ['USOIL','XAUUSD','USDCAD','SPX']:
    b,c,net=direcao(s,temas)
    print(f"  {s}: {({1:'COMPRA',-1:'VENDA',0:'NEUTRO'}[b])}  (conf {c:.2f}, net {net:+.2f})")


## Passo 5 — Plugar no bot (depois)
1. Baixe `/content/finbert_ft`. 2. Em `NewsBiasEngine._score_sentiment` carregue o modelo
(igual Passo 2) e em `_interpret_macro` use a lógica do Passo 4 (ou o `news_mapper.py`).
3. `USE_AI=true` no `.env` e rode na **DEMO** primeiro.
